# ДЗ 2 

Задача: бинарная классификация на наличие пневнмонии на датасете [Chest X-Ray Images](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)  

## 1. Настройка окружения

In [ ]:
import warnings

import os
import random
import collections
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import kagglehub

from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.amp.autocast_mode import autocast
from torch.amp.grad_scaler import GradScaler
from datasets import load_dataset, concatenate_datasets
from sklearn.metrics import (
    balanced_accuracy_score, classification_report,
    f1_score, recall_score, ConfusionMatrixDisplay
)

from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

for w in [UserWarning, FutureWarning]:
    warnings.filterwarnings("ignore", category=w)


In [ ]:
SEED = 10091991
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

## 2. Загрузка данных

In [ ]:
kaggle_path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
data_dir = os.path.join(kaggle_path, "chest_xray", "chest_xray")
dataset = load_dataset("imagefolder", data_dir=data_dir)

### Пересборка train/val split

Оригинальный validation split содержит всего 16 изображений.  
Объединяем train + val и делаем стратифицированное разбиение с размером валидационной выборки в 15%.

In [ ]:
merged = concatenate_datasets([dataset["train"], dataset["validation"]])
split = merged.train_test_split(test_size=0.15, seed=42, stratify_by_column="label")

dataset_train = split["train"]
dataset_val   = split["test"]
dataset_test  = dataset["test"]

print(f"Train: {len(dataset_train)}, Val: {len(dataset_val)}, Test: {len(dataset_test)}")

## 3. Предобработка данных

Рентгеновские снимки — монохромные, поэтому используем 1 канал.  
Для train применим flip, rotation, jitter для регуляризации.  
Для val/test — только resize и нормализация.  

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=1),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])


def preprocess(transform_fn):
    def _preproc(batch):
        batch["pixel_values"] = [transform_fn(img.convert("L")) for img in batch["image"]]
        batch["label"] = batch["label"]
        return batch
    return _preproc


dataset_train.set_transform(preprocess(train_transform))
dataset_val.set_transform(preprocess(transform))
dataset_test.set_transform(preprocess(transform))

In [ ]:
def collate_fn(batch):
    pixels = torch.stack([x["pixel_values"] for x in batch])
    labels = torch.tensor([x["label"] for x in batch])
    return pixels, labels


train_loader = DataLoader(dataset_train, batch_size=64, shuffle=True,
                          collate_fn=collate_fn, num_workers=0, pin_memory=True)
val_loader = DataLoader(dataset_val, batch_size=64, shuffle=False,
                        collate_fn=collate_fn, num_workers=0, pin_memory=True)

In [ ]:
dataset_train.reset_format()
counts = collections.Counter(dataset_train["label"])
dataset_train.set_transform(preprocess(train_transform))

print(f"Normal: {counts[0]}, Pneumonia: {counts[1]}, ratio: {counts[1]/counts[0]:.2f}:1")
pos_weight = torch.tensor([counts[0] / counts[1]])
print(f"pos_weight: {pos_weight.item():.4f}")

Заметим, что соотношение классов не равное. Нужно использовать pos_weight в функции потерь. 

Undersampling делать не стал так как датасет уже маленький

## 4. Архитектура модели

### Baseline: CNN

Три свёрточных блока (Conv -> BatchNorm -> ReLU -> MaxPool) с каналами 32 -> 64 -> 128.  
AdaptiveAvgPool2d(1) делает классификатор независимым от входного разрешения.  
Один выход + BCEWithLogitsLoss для бинарной классификации.

### Улучшение: ResBlock

Добавляем Residual Blocks (skip connections) после каждого свёрточного блока.  
Это позволяет увеличить глубину сети без проблемы затухающих градиентов.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        return F.relu(self.block(x) + x)


class CNNBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(-1)


class CNNRes(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            ResBlock(32),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            ResBlock(64),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            ResBlock(128),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(-1)


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

model = CNNRes().to(device)
pos_weight = pos_weight.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Device: {device}")
print(f"Parameters: {total_params:,}")

## 5. Обучение модели

- Оптимизатор: AdamW 
- Функция потерь: BCEWithLogitsLoss с pos_weight для компенсации дисбаланса
- Scheduler: ReduceLROnPlateau — снижает LR при стагнации val F1
- Метрика для model selection: F1 Macro — учитывает и precision, и recall по обоим классам

In [ ]:
writer = SummaryWriter("runs/")
model = CNNRes().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
scaler = GradScaler()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.5)

best_val_f1 = 0.0
num_epochs = 50

In [ ]:
@dataclass
class ModelMetrics:
    loss: float
    bacc: float
    f1: float
    recall_normal: float
    recall_pneumonia: float

In [ ]:
def count_stats(running_loss, preds, labels) -> ModelMetrics:
    train_loss = running_loss / len(labels)
    train_bacc = balanced_accuracy_score(labels, preds)
    train_f1 = f1_score(labels, preds, average="macro")
    train_recall_normal = recall_score(labels, preds, pos_label=0)
    train_recall_pneumonia = recall_score(labels, preds, pos_label=1)

    return ModelMetrics(
        loss=float(train_loss),
        bacc=float(train_bacc),
        f1=float(train_f1),
        recall_normal=float(train_recall_normal),
        recall_pneumonia=float(train_recall_pneumonia),
    )

In [ ]:
def train_one_epoch() -> ModelMetrics:
    model.train()
    running_loss = 0.0
    train_preds, train_labels = [], []

    for images, labels in train_loader:
        images, labels = images.to(device), labels.float().to(device)
        optimizer.zero_grad()
        with autocast(device_type=device.type):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * len(labels)
        preds = (torch.sigmoid(logits) > 0.5).long()
        train_preds.extend(preds.cpu().tolist())
        train_labels.extend(labels.long().cpu().tolist())

    return count_stats(running_loss, train_preds, train_labels)

In [ ]:
def eval_model() -> tuple[ModelMetrics, list[int], list[int]]:
    model.eval()
    val_loss_sum = 0.0
    val_preds, val_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.float().to(device)
            logits = model(imgs)
            val_loss_sum += criterion(logits, labels).item() * len(labels)
            preds = (torch.sigmoid(logits) > 0.5).long()
            val_preds.extend(preds.cpu().tolist())
            val_labels.extend(labels.long().cpu().tolist())

    metrics = count_stats(val_loss_sum, val_preds, val_labels)
    return metrics, val_labels, val_preds

In [ ]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    train_preds, train_labels = [], []

    train_metrics = train_one_epoch()

    # validation
    model.eval()
    val_metrics, val_labels, val_preds = eval_model()
    scheduler.step(val_metrics.f1)

    # TensorBoard
    writer.add_scalars("Loss", {"train": train_metrics.loss, "val": val_metrics.loss}, epoch)
    writer.add_scalars("Balanced Accuracy", {"train": train_metrics.bacc, "val": val_metrics.bacc}, epoch)
    writer.add_scalars("F1 Macro", {"train": train_metrics.f1, "val": val_metrics.f1}, epoch)
    writer.add_scalars("Recall Normal", {"train": train_metrics.recall_normal, "val": val_metrics.recall_normal}, epoch)
    writer.add_scalars("Recall Pneumonia", {"train": train_metrics.recall_pneumonia, "val": val_metrics.recall_pneumonia}, epoch)

    # save best
    is_best = val_metrics.f1 > best_val_f1
    if is_best:
        best_val_f1 = val_metrics.f1
        torch.save(model.state_dict(), "best_model.pt")

    print(f"Epoch {epoch+1:02d}/{num_epochs} — "
          f"loss: {train_metrics.loss:.4f}/{val_metrics.loss:.4f}, "
          f"bacc: {train_metrics.bacc:.4f}/{val_metrics.bacc:.4f}, "
          f"f1: {train_metrics.f1:.4f}/{val_metrics.f1:.4f}, "
          f"rec_N: {val_metrics.recall_normal:.4f}, rec_P: {val_metrics.recall_pneumonia:.4f}"
          + (" +" if is_best else ""))

writer.close()

print(f"\nBest val F1 (macro): {best_val_f1:.4f}")
print(classification_report(val_labels, val_preds, target_names=["Normal", "Pneumonia"]))

## 6. Визуализация метрик обучения

Графики loss и F1 из TensorBoard логов для последнего эксперимента.

In [ ]:
def read_scalar(path):
    ea = EventAccumulator(path)
    ea.Reload()
    tag = ea.Tags()["scalars"][0]
    return [(s.step, s.value) for s in ea.Scalars(tag)]

loss_train = read_scalar("runs/Loss_train")
loss_val = read_scalar("runs/Loss_val")
f1_train = read_scalar("runs/F1 Macro_train")
f1_val = read_scalar("runs/F1 Macro_val")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(*zip(*loss_train), label="Train")
ax1.plot(*zip(*loss_val), label="Val")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.legend()

ax2.plot(*zip(*f1_train), label="Train")
ax2.plot(*zip(*f1_val), label="Val")
ax2.set_title("F1 Macro")
ax2.set_xlabel("Epoch")
ax2.legend()

plt.tight_layout()
plt.show()

## 7. Оценка на тестовой выборке (Лушая по F1 модель)


In [ ]:
# load best model
test_loader = DataLoader(dataset_test, batch_size=64, shuffle=False, collate_fn=collate_fn)
model.load_state_dict(torch.load("best_model.pt", map_location=device))
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        logits = model(imgs.to(device))
        preds = (torch.sigmoid(logits) > 0.5).long()
        test_preds.extend(preds.cpu().tolist())
        test_labels.extend(labels.tolist())

print(classification_report(test_labels, test_preds, target_names=["Normal", "Pneumonia"]))
print(f"Balanced Accuracy: {balanced_accuracy_score(test_labels, test_preds):.4f}")
print(f"F1 Macro: {f1_score(test_labels, test_preds, average='macro'):.4f}")

### Матрица ошибок

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    test_labels, test_preds,
    display_labels=["Normal", "Pneumonia"],
    cmap="Blues", ax=ax
)
ax.set_title("Test Set Confusion Matrix")
plt.tight_layout()
plt.show()

### Правильные и ошибочные предсказания

Верхний ряд - корректно классифицированные снимки, нижний - ошибки модели.

In [ ]:
model.eval()
samples = []
with torch.no_grad():
    for imgs, labels in test_loader:
        probs = torch.sigmoid(model(imgs.to(device))).cpu()
        for img, label, prob in zip(imgs, labels, probs):
            pred = 1 if prob > 0.5 else 0
            samples.append((img, label.item(), pred, prob.item()))

correct = [s for s in samples if s[1] == s[2]]
wrong = [s for s in samples if s[1] != s[2]]

names = ["Normal", "Pneumonia"]

fig, axes = plt.subplots(2, 5, figsize=(15, 7))
fig.suptitle("Top: Correct | Bottom: Incorrect", fontsize=14)

for i, s in enumerate(random.sample(correct, 5)):
    img, label, pred, prob = s
    axes[0, i].imshow(img.squeeze(), cmap="gray")
    axes[0, i].set_title(f"True: {names[label]}\nP(pneu): {prob:.2f}", fontsize=9, color="green")
    axes[0, i].axis("off")

for i, s in enumerate(random.sample(wrong, min(5, len(wrong)))):
    img, label, pred, prob = s
    axes[1, i].imshow(img.squeeze(), cmap="gray")
    axes[1, i].set_title(f"True: {names[label]}, Pred: {names[pred]}\nP(pneu): {prob:.2f}", fontsize=9, color="red")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

## 8. Демонстрация работы модели

In [ ]:
model = CNNRes()
model.load_state_dict(torch.load("best_model.pt", map_location="cpu"))
model.eval()

fig, axes = plt.subplots(1, 5, figsize=(15, 4))
names = ["Normal", "Pneumonia"]

for i, idx in enumerate(random.sample(range(len(dataset_test)), 5)):
    sample = dataset_test[idx]
    img = sample["pixel_values"]
    label = sample["label"]
    prob = torch.sigmoid(model(img.unsqueeze(0))).item()
    pred = 1 if prob > 0.5 else 0

    axes[i].imshow(img.squeeze(), cmap="gray")
    axes[i].set_title(f"True: {names[label]}\nPred: {names[pred]} ({prob:.2f})", fontsize=10)
    axes[i].axis("off")

plt.suptitle("Model Demo — Predictions with Probabilities")
plt.tight_layout()
plt.show()

## 9. Сводка экспериментов

| Эксперимент | Val F1 | Комментарий |
|---|---|---|
| Baseline CNN 16→32→64 | 0.926 | Переобучение |
| + Аугментации | 0.878 | Недообучение   |
| + ResBlocks 32→64→128 | 0.918 | Стабильнее, но всё ещё не сходится |
| + 50 эпох | **0.970** | Лучший результат |
| + SE Blocks | 0.964 | Без значимого улучшения, даже чуть хуже |
| Threshold tuning (0.44) | — | Не перенёсся на тест (domain shift) |


## 10. Выводы

### Достигнутые метрики
На валидации модель ведет себя достаточно прилично f1 = 0.97

Но на тесте все портится: f1 падает до 0.83. Как можно видеть по confusion matrix, 
много ложно позитивных предсказаний. Скорее это связано с разным распределением.
классов на train / test. 

### Что работает

- Residual connections основной вклад в качество (скачок с 0.926 до 0.970)
- Data augmentation  стабилизирует обучение и уменьшает разрыв train/val
- `pos_weight` простой и эффективный способ компенсации дисбаланса классов
- `ReduceLROnPlateau` автоматическая адаптация learning rate

### Проблемы
Как видно, у нас произошло падение метрики f1 из-за роста количества ложно-позитивных предсказаний. 
Возможно модель не смогла хорошо обобщиться из-за невыроженности нормального класса в обучающей выбоке.

### Возможные улучшения
- Transfer learning 
- Больше данных в датасет чтобы убрать имбаланс классов.